# OpenVLA resident adapter (`--backend openvla`)

Serves OpenVLA (the original 7B model) from its own resident Flask adapter, `control/vla_bridge/openvla_server.py`, through an ngrok HTTP tunnel.

| | |
|---|---|
| Lane | HTTP adapter (isolated Python 3.10 venv) |
| Port → tunnel | 3003 → ngrok HTTP |
| GPU runtime | L4 |
| Gated HF licences | none |
| Sign-offs before running | 14-UAT 5 licence decision |
| Default checkpoint | `toasta18/openvla-so101-cylinder@f00833ce5851` |
| Server log | `/content/openvla_server.log` |
| Embedded files (written to `/content`) | `colab_server_utils.py`, `openvla_server.py` |

Run top to bottom on a fresh runtime. Works from the VS Code Colab extension: nothing to upload.
Every file this notebook runs is written into `/content` by a cell below and checked against
its sha256. Run the teardown cell at the end of every session. The index of all notebooks,
with the local smoke and episode commands, is `control/vla_bridge/notebooks/README.md`.

Generated by `scripts/build_colab_notebooks.py` from `scripts/colab_notebook_specs.py` +
`control/vla_bridge` sources, so edit those and regenerate, not this file.

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

**pyngrok legitimacy gate:** the next cell installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

In [ ]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

In [ ]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

## Server helper

The next cell writes `control/vla_bridge/colab_server_utils.py` into `/content/colab_server_utils.py`; the cell after it checks its sha256. The launch, diagnostic and teardown cells import it to stop stale servers, prove this notebook's server owns its port, and report logs, processes and GPU memory.

In [ ]:
%%writefile /content/colab_server_utils.py
"""Colab-side process/port helper for the per-backend policy-server notebooks.

Every notebook in control/vla_bridge/notebooks/ writes this file to
/content/colab_server_utils.py with a %%writefile cell (byte-identical to this file, checked
by a sha256 cell and by control/test_colab_notebooks.py) and imports it with
sys.path.insert(0, '/content'). Local control code never imports it.

What it guarantees for a launch cell:
- stop_servers() clears any earlier server: every process whose cmdline contains a known
  server marker, plus whatever is listening on the notebook's port.
- start_server() starts the server unbuffered (PYTHONUNBUFFERED=1) with its log in a file.
- wait_until_ready() only succeeds when the process this cell started (or one of its
  children) owns the port. A stale server that answers on the port is an error, not a
  success. It also fails on an early exit and, for patched launchers, on a missing banner.

The diagnostic helpers (log_report, process_report, gpu_report, health_report) print what a
partner needs to debug a run, and stop_server() is the teardown.

Imports: stdlib + psutil only (psutil is preinstalled on Colab).
"""

import json
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Callable, Mapping, Optional, Sequence

import psutil

SERVER_MARKERS = (
    "lerobot.async_inference.policy_server",
    "diffusion_policy_server.py",
    "groot_n17_server.py",
    "fastfs_server.py",
    "openvla_oft_server.py",
    "openvla_server.py",
    "octo_server.py",
    "wallx_server.py",
    "rdt1b_server.py",
    "groot_n15_server.py",
)

# Case-sensitive on purpose: "Error" covers KeyError, NotImplementedError, GatedRepoError and
# "Error in StreamActions"; lowercase prose such as "error budget" is not a match.
ERROR_PATTERN = re.compile(
    r"ERROR|Error|Traceback|Exception|OutOfMemory|out of memory|Killed|beginAllocateToPool"
)

_ACCESS_ERRORS = (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess)


def _cmdline(proc) -> str:
    try:
        return " ".join(proc.cmdline())
    except _ACCESS_ERRORS:
        return ""
    except OSError:
        return ""


def _describe(pid: int) -> str:
    try:
        return f"PID {pid} ({_cmdline(psutil.Process(pid)) or '?'})"
    except _ACCESS_ERRORS:
        return f"PID {pid} (gone)"


def _tcp_connections(proc):
    # Colab's psutil may predate 6.0, where net_connections() replaced connections().
    method = getattr(proc, "net_connections", None) or proc.connections
    return method(kind="tcp")


def _read_lines(log_path) -> list:
    path = Path(log_path)
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()


def _tail(log_path, lines: int = 40) -> str:
    return "\n".join(_read_lines(log_path)[-lines:])


def listening_pids(port: int) -> set:
    """PIDs of every visible process with a TCP socket in LISTEN state on `port`."""
    pids = set()
    for proc in psutil.process_iter():
        try:
            for conn in _tcp_connections(proc):
                if conn.status == psutil.CONN_LISTEN and conn.laddr and conn.laddr.port == port:
                    pids.add(proc.pid)
        except _ACCESS_ERRORS:
            continue
        except OSError:
            continue
    return pids


def stop_servers(port: int, patterns: Sequence[str] = (), timeout_s: float = 15.0) -> list:
    """Stop every process matching a pattern plus the port's listener; return what was stopped."""
    me = os.getpid()
    targets = {}
    for proc in psutil.process_iter():
        if proc.pid == me:
            continue
        cmdline = _cmdline(proc)
        if cmdline and any(pattern and pattern in cmdline for pattern in patterns):
            targets[proc.pid] = proc
    for pid in listening_pids(port):
        if pid != me and pid not in targets:
            try:
                targets[pid] = psutil.Process(pid)
            except _ACCESS_ERRORS:
                continue
    stopped = []
    for pid, proc in sorted(targets.items()):
        description = f"PID {pid} ({_cmdline(proc) or '?'})"
        try:
            proc.terminate()
            stopped.append(description)
        except psutil.NoSuchProcess:
            continue
        except psutil.AccessDenied:
            print(f"Cannot stop {description}: access denied")
    _, alive = psutil.wait_procs(list(targets.values()), timeout=5)
    for proc in alive:
        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=5)
    for line in stopped:
        print(f"Stopped {line}")
    deadline = time.monotonic() + timeout_s
    while True:
        holders = listening_pids(port) - {me}
        if not holders:
            return stopped
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"port {port} is still held by "
                + ", ".join(_describe(pid) for pid in sorted(holders))
                + f" after {timeout_s:.0f} s; stop it by hand before launching"
            )
        time.sleep(0.2)


def start_server(argv: Sequence[str], log_path,
                 env_overrides: Optional[Mapping[str, str]] = None,
                 cwd=None) -> subprocess.Popen:
    """Start `argv` unbuffered with stdout+stderr in `log_path` (truncated)."""
    env = os.environ.copy()
    env.update(env_overrides or {})
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "wb", buffering=0)
    try:
        proc = subprocess.Popen(list(argv), stdout=log, stderr=subprocess.STDOUT, env=env,
                                cwd=None if cwd is None else str(cwd))
    finally:
        log.close()
    print(f"Started PID {proc.pid}: {' '.join(argv)}")
    print(f"Log → {log_path}")
    return proc


def _probe(port: int, health_url: Optional[str], ready: Optional[Callable[[dict], bool]]):
    """Return (answered, health): answered is True once the server counts as ready."""
    if health_url:
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                health = json.loads(response.read().decode())
        except Exception:
            return False, None
        if not isinstance(health, dict):
            return False, None
        ok = ready(health) if ready is not None else health.get("status") == "ready"
        return bool(ok), health
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True, None
    except OSError:
        return False, None


def _own_pids(proc) -> set:
    pids = {proc.pid}
    try:
        pids.update(child.pid for child in psutil.Process(proc.pid).children(recursive=True))
    except _ACCESS_ERRORS:
        pass
    return pids


def wait_until_ready(proc, port: int, log_path, timeout_s: float,
                     health_url: Optional[str] = None,
                     ready: Optional[Callable[[dict], bool]] = None,
                     banner: Optional[str] = None,
                     settle_s: float = 2.0, poll_s: float = 1.0):
    """Wait until the server this cell started owns `port` and is ready.

    Returns the /health dict in health mode, None in TCP mode. Raises RuntimeError on an
    early exit, a foreign/stale owner or a missing launcher banner, and TimeoutError (after
    terminating `proc`) when the deadline passes.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        code = proc.poll()
        if code is not None:
            raise RuntimeError(
                f"server PID {proc.pid} exited with code {code} before it was ready. "
                f"Last log lines of {log_path}:\n{_tail(log_path)}"
            )
        answered, health = _probe(port, health_url, ready)
        if answered:
            owners = listening_pids(port)
            mine = _own_pids(proc)
            if not owners or not owners <= mine:
                foreign = ", ".join(_describe(pid) for pid in sorted(owners - mine))
                foreign = foreign or "no visible process"
                raise RuntimeError(
                    f"port {port} answered but is owned by {foreign}, not the server this cell "
                    f"started (PID {proc.pid}). That is a stale server; re-run the launch cell, "
                    "which stops old servers first."
                )
            time.sleep(settle_s)
            code = proc.poll()
            if code is not None:
                raise RuntimeError(
                    f"server PID {proc.pid} exited with code {code} right after binding port "
                    f"{port}. Last log lines of {log_path}:\n{_tail(log_path)}"
                )
            if banner is not None:
                text = Path(log_path).read_text(errors="replace") if Path(log_path).exists() else ""
                if banner not in text:
                    raise RuntimeError(
                        f"expected launcher banner '{banner}' not found in {log_path}, so this "
                        "is not the patched launcher. Re-run the embed cells, then the launch cell."
                    )
            print(f"READY: PID {proc.pid} ({_cmdline(psutil.Process(proc.pid))}) owns port "
                  f"{port}; log → {log_path}")
            return health
        time.sleep(poll_s)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    raise TimeoutError(
        f"server PID {proc.pid} was not ready on port {port} within {timeout_s:.0f} s "
        f"(terminated). Last log lines of {log_path}:\n{_tail(log_path)}"
    )


def log_report(log_path, tail: int = 80, max_matches: int = 40) -> None:
    """Print the last `tail` log lines and up to `max_matches` numbered error lines."""
    path = Path(log_path)
    if not path.exists():
        print(f"{path} does not exist (has the launch cell run on this runtime?)")
        return
    lines = _read_lines(path)
    print(f"--- last {tail} lines of {path} ({len(lines)} lines total) ---")
    for line in lines[-tail:]:
        print(line)
    matches = [(number, line) for number, line in enumerate(lines, start=1)
               if ERROR_PATTERN.search(line)]
    print(f"--- {len(matches)} lines matching ERROR_PATTERN (showing up to {max_matches}) ---")
    for number, line in matches[:max_matches]:
        print(f"{number}: {line}")
    if not matches:
        print("(none)")


def process_report(port: int, expected_marker: str, proc=None) -> None:
    """Print every known server process, then check the right server owns `port`."""
    print("--- running server processes ---")
    found = False
    now = time.time()
    for candidate in psutil.process_iter():
        cmdline = _cmdline(candidate)
        if not cmdline or not any(marker in cmdline for marker in SERVER_MARKERS):
            continue
        try:
            elapsed = now - candidate.create_time()
            rss_mb = candidate.memory_info().rss / 1e6
        except _ACCESS_ERRORS:
            continue
        found = True
        print(f"PID {candidate.pid}  up {elapsed:7.0f} s  RSS {rss_mb:8.1f} MB  {cmdline}")
    if not found:
        print("(no process with a known server marker)")
    owners = listening_pids(port)
    if not owners:
        raise RuntimeError(f"nothing is listening on port {port}; run the launch cell")
    for pid in sorted(owners):
        try:
            cmdline = _cmdline(psutil.Process(pid))
        except _ACCESS_ERRORS:
            cmdline = ""
        if expected_marker not in cmdline:
            raise RuntimeError(
                f"WRONG server on port {port}: PID {pid} ({cmdline or '?'}) does not contain "
                f"'{expected_marker}'. Run the teardown cell, then the launch cell."
            )
    if proc is not None:
        mine = _own_pids(proc)
        if not owners <= mine:
            raise RuntimeError(
                f"port {port} is owned by "
                + ", ".join(_describe(pid) for pid in sorted(owners))
                + f", not by the server this notebook started (PID {proc.pid})"
            )
    print(f"OK: {expected_marker} (PID {min(owners)}) owns port {port}")


def gpu_report() -> None:
    """Print GPU memory/utilisation and the processes using the GPU."""
    queries = (
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv"],
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    )
    for argv in queries:
        try:
            result = subprocess.run(argv, capture_output=True, text=True, timeout=30)
        except FileNotFoundError:
            print("no nvidia-smi (CPU runtime?): pick a GPU runtime for this notebook")
            return
        print(result.stdout.rstrip() or result.stderr.rstrip())


def health_report(port: int) -> dict:
    """GET /health on the local server, pretty-print it and return it."""
    url = f"http://127.0.0.1:{port}/health"
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health = json.loads(response.read().decode())
    except Exception as exc:
        print(f"GET {url} failed: {type(exc).__name__}: {exc}")
        raise
    print(json.dumps(health, indent=2, sort_keys=True))
    return health


def stop_server(proc, port: int, patterns: Sequence[str] = ()) -> None:
    """Teardown: stop `proc` (if alive) and any leftover server, then confirm the port is free."""
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
        print(f"Stopped PID {proc.pid}")
    stop_servers(port, patterns)
    print(f"port {port} is free")


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2d8f63cbfdf5e679c268dd4f4436129e8fa8f30acfd5c5f5d88eb1082c2bf2a8'
actual = hashlib.sha256(Path('/content/colab_server_utils.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/colab_server_utils.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/colab_server_utils.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/colab_server_utils.py matches control/vla_bridge/colab_server_utils.py (sha256 {actual[:12]})')

## Tier C: OpenVLA resident adapter (Plan 14-09, `--backend openvla`)

OpenVLA (the original 7B model, not OFT) has no LeRobot `PolicyServer` integration. It runs as its
own resident Flask adapter, `control/vla_bridge/openvla_server.py`, built like the OpenVLA-OFT
adapter (see README): it loads once, warms up, then serves. It does not use the gRPC PolicyServer lane.

Unlike OpenVLA-OFT, it needs **no GitHub clone**. The official inference path
(github.com/openvla/openvla README and `vla-scripts/deploy.py`) is plain
`transformers` AutoClasses with `trust_remote_code=True`. The adapter pins that remote code: it
copies `openvla/openvla-7b@47a0ec7f`'s three `*_prismatic.py` files into the pinned checkpoint
snapshot and rewrites `auto_map` to point at them, so no other code can run.

**Status:** written from upstream source and HF Hub metadata. These cells have **not yet been run
on Colab**.

Requirements:
- **GPU:** an A100 or L4. The model is 7.5B parameters in bf16, about 15 GB of weights, so it
  needs about 16 GB of VRAM.
- **Dependencies:** OpenVLA's modeling code requires `timm` 0.9.x and expects
  `transformers==4.40.1` / `tokenizers==0.19.1`. Those must never touch the PolicyServer's
  environment, so the adapter runs in its own Python 3.10 venv. `openvla_server.py` refuses to
  start on other versions.

Default checkpoint: `toasta18/openvla-so101-cylinder@f00833ce5851`. It is an SO-101 joint-space
fine-tune (degrees, gripper percent) with the single task "Pick up the black cylinder and place it
on the plate." The official MIT `openvla/openvla-7b-finetuned-libero-{object,spatial,goal,10}`
checkpoints are reviewed alternates: their delta-EEF output goes through IK on the control side.

In [ ]:
# OpenVLA only: an isolated Python 3.10 venv with the versions OpenVLA pins (README /
# pyproject / modeling_prismatic.py). No GitHub clone, no flash-attn build (the adapter uses sdpa).
# Both the install and the import check are verified by exit status.
import subprocess, sys
from pathlib import Path
OPENVLA_VENV = '/content/openvla_venv'
OPENVLA_PYTHON = f'{OPENVLA_VENV}/bin/python'
if not Path(OPENVLA_PYTHON).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
    subprocess.run(['uv', 'venv', '--python', '3.10', '--seed', OPENVLA_VENV], check=True)
install = subprocess.run(
    ['uv', 'pip', 'install', '--python', OPENVLA_PYTHON,
     'torch==2.2.0', 'torchvision==0.17.0', 'transformers==4.40.1', 'tokenizers==0.19.1',
     'timm==0.9.10', 'accelerate==0.30.1', 'sentencepiece==0.1.99', 'numpy<2',
     'pillow', 'flask', 'opencv-python-headless', 'requests'],
    capture_output=True, text=True)
print(install.stdout[-2000:]); print(install.stderr[-2000:])
if install.returncode != 0:
    raise RuntimeError(f'OpenVLA venv install failed (exit {install.returncode})')
check = subprocess.run([OPENVLA_PYTHON, '-c', (
    "import importlib.metadata as m, torch\n"
    "print({p: m.version(p) for p in ('torch', 'transformers', 'tokenizers', 'timm', 'accelerate', 'huggingface-hub')})\n"
    "assert torch.cuda.is_available(), 'no CUDA GPU visible to the OpenVLA venv'\n"
)], capture_output=True, text=True)
print(check.stdout); print(check.stderr[-2000:])
if check.returncode != 0:
    raise RuntimeError('OpenVLA venv import check failed; see stderr above')

## Adapter source and launch notes

The next cell writes `control/vla_bridge/openvla_server.py` into `/content/openvla_server.py`, and the cell after it checks its sha256. The launch cell then starts it as a separate OS process.

It runs with the venv's Python.

On first start it does two downloads, about 15 GB in total:
- the pinned checkpoint revision
- the three pinned `openvla/openvla-7b` code files

`/health` only answers after the model has loaded and warmed up. It reports `unnorm_key`,
`action_space` and the action q01/q99. Logs go to `/content/openvla_server.log`.

To use a LIBERO alternate, add `--checkpoint openvla/openvla-7b-finetuned-libero-object` to the
launch cell's command. Its reviewed revision and unnorm key are applied automatically. Pass the
**same** `--checkpoint` to `run_vla_episode.py`, because it selects the client-side conversion.

In [ ]:
%%writefile /content/openvla_server.py
"""Resident OpenVLA HTTP adapter (Tier C, Plan 14-09). Run on Colab inside the
isolated OpenVLA venv (see vla_bridge/notebooks/http_openvla.ipynb).

Same shape as `openvla_oft_server.py` and `fastfs_server.py`:
- the model loads ONCE
- one throwaway warmup inference runs before the socket opens
- `/health` reports ready
- a non-blocking `inference_lock` answers concurrent requests with 429
- `ValueError`/`KeyError`/`TypeError` on a request become 400
- a model failure (`RuntimeError`, e.g. a non-finite action) becomes 500,
  so the client holds position instead of parsing NaN JSON

The server stays model-format-only. It returns OpenVLA's raw 7-D
`predict_action()` output, and `backends/openvla.py` turns that into a
`JOINT_ORDER` target for the checkpoint's action space.

Verified against live upstream source (2026-10-01, execution discipline)
------------------------------------------------------------------------
- Loading: OpenVLA (not OFT) needs no repo install. github.com/openvla/openvla
  @ c8f03f4 (`README.md` quick start, `vla-scripts/deploy.py`,
  `experiments/robot/openvla_utils.get_vla/get_processor/get_vla_action`) loads
  with `AutoProcessor.from_pretrained(path, trust_remote_code=True)` and
  `AutoModelForVision2Seq.from_pretrained(path, torch_dtype=bfloat16,
  low_cpu_mem_usage=True, trust_remote_code=True)`, then calls
  `vla.predict_action(**processor(prompt, image), unnorm_key=..., do_sample=False)`.
- Prompt: `"In: What action should the robot take to {instruction.lower()}?\\nOut:"`
  (deploy.py `get_openvla_prompt`; the fine-tune's `PurePromptBuilder`).
- One image only. `predict_action` takes a single third-person image. The LIBERO
  eval passes `full_image` only; finetune.py loads `load_camera_views=("primary",)`.
- `predict_action` (modeling_prismatic.py) decodes 256-bin tokens and
  un-normalizes with the chosen key's q01/q99 where `mask` is true. Masked-off
  dims come back as the raw bin centre in [-1, 1].
- Center crop: the LIBERO checkpoints and finetune.py's default `image_aug=True`
  train with random crops, so evaluation center-crops `sqrt(0.9)` of each side and
  resizes back to 224 (`openvla_utils.crop_and_resize`). That is reproduced here
  with PIL (a bilinear box resize), so TensorFlow isn't needed. The pixels differ
  slightly from TF's `crop_and_resize`.
- Remote code pin (T-14-06): the fine-tuned checkpoints ship NO
  `*_prismatic.py`. Their `config.json` / `preprocessor_config.json` /
  `tokenizer_config.json` `auto_map` point at
  `openvla/openvla-7b--<module>.<Class>`. transformers 4.40.1
  (`dynamic_module_utils.get_class_from_dynamic_module`) resolves that
  cross-repo code at `code_revision`, which `AutoProcessor` drops before its
  inner `AutoImageProcessor` call, so it would load `main`. Instead
  `_localize_remote_code` copies the three files from openvla/openvla-7b at the
  reviewed sha into the pinned snapshot and rewrites `auto_map` to local
  module names. `trust_remote_code` then only ever executes that reviewed code.
  The sha is 47a0ec7f. Its only commit after 31f090d ("Update modeling_prismatic.py
  to match public GitHub repo") is a README edit. Its `modeling_prismatic.py` is
  byte-identical to github.com/openvla/openvla @ c8f03f4's
  `prismatic/extern/hf/modeling_prismatic.py`.
- Dependency pins: `modeling_prismatic.py` raises unless `timm` is 0.9.10-0.9.16 and
  warns unless `transformers==4.40.1` / `tokenizers==0.19.1` (the README cites
  inference regressions on other versions). `main()` refuses to start otherwise.

Checkpoints (HF API, read 2026-10-01; see `CHECKPOINT_PROFILES`)
----------------------------------------------------------------
- Default: `toasta18/openvla-so101-cylinder` @ f00833c. It is a full OpenVLA-7B
  fine-tune on a real SO-101 (`so_follower`) lerobot dataset,
  `toasta18/so101_task_yellow_cylinder`. That dataset has 50 episodes, 30 fps,
  `overhead` + `wrist` cameras, and the single task "Pick up the black cylinder and
  place it on the plate.". It was converted to RLDS (`toasta18/so101_pick_cylinder_rlds`:
  `image`, `wrist_image`, a 7-D action).
  Its `so101_pick_cylinder` action stats are ABSOLUTE joint positions in degree
  mode: slots 0-4 are the arm joints in JOINT_ORDER, slot 5 is a constant 0 pad,
  and slot 6 is the gripper percent. So no IK is needed. Because its config embeds the
  25 base OXE keys as well, the unnorm key must be explicit.
- Alternates: the official MIT `openvla/openvla-7b-finetuned-libero-{object,
  spatial,goal,10}`. They emit robosuite OSC_POSE deltas, and the client converts
  those through `eef_to_joint.solve_ik()`.

Known limitations (documented, not hidden)
------------------------------------------
- toasta18's model card is empty and its repo has no license. Its config's action
  `mask` was edited after upload (commits a545284 -> f00833c) to
  `[T,T,T,T,T,F,T]`. Its own `dataset_statistics.json` still records OpenVLA's forced
  EEF_POS training mask `[T]*6+[F]`
  (`materialize.make_oxe_dataset_kwargs`). Under that mask the gripper target
  (1-53 %) was never normalized, so the action tokenizer clipped it to 1.0. The
  gripper output is therefore expected to sit near q99 (about 52.7 %) whatever the
  scene. The arm joints were trained normally. Inferred from the published
  stats, not from running the model.
- Which RLDS camera (`image`) the fine-tune saw is assumed to be the dataset's
  `overhead` view (LIBERO-style naming). The default `--image-key` is therefore
  `overhead_left`, one half of this rig's AR0144 stereo pair.
- The LIBERO checkpoints are Franka-sim policies, so those are out of distribution
  on SO-101 hardware.
- Each request is stateless. OpenVLA predicts one action per call, with no chunk.

Colab launch: see vla_bridge/notebooks/http_openvla.ipynb.
"""
import argparse
import base64
import json
import math
import shutil
import threading
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

DEFAULT_CHECKPOINT = "toasta18/openvla-so101-cylinder"
DEFAULT_REVISION = "f00833ce58510a955b981ee7e9ba8e9ef4e2ef2e"
DEFAULT_LOCAL_DIR = "/content/openvla_checkpoints"
DEFAULT_IMAGE_KEY = "overhead_left"
DEFAULT_PORT = 3003

# T-14-06: the only code trust_remote_code may execute (HF API sha, read 2026-10-01).
CODE_REPO = "openvla/openvla-7b"
CODE_REVISION = "47a0ec7fc4ec123775a391911046cf33cf9ed83f"
REMOTE_CODE_FILES = ("configuration_prismatic.py", "modeling_prismatic.py", "processing_prismatic.py")
AUTO_MAP_FILES = ("config.json", "preprocessor_config.json", "processor_config.json", "tokenizer_config.json")
_LOCAL_MODULES = tuple(name[: -len(".py")] for name in REMOTE_CODE_FILES)

# Reviewed checkpoints. `backend_configs/openvla.json["checkpoints"]` must agree
# (pinned by test_client_and_server_checkpoint_profiles_agree).
CHECKPOINT_PROFILES = {
    "toasta18/openvla-so101-cylinder": dict(
        revision="f00833ce58510a955b981ee7e9ba8e9ef4e2ef2e", unnorm_key="so101_pick_cylinder",
        action_space="absolute_joint_degrees_so101_7d"),
    "openvla/openvla-7b-finetuned-libero-object": dict(
        revision="287d6cfdf12d07b1449505f66d9bf3550257e9b3", unnorm_key="libero_object",
        action_space="delta_eef_robosuite_osc"),
    "openvla/openvla-7b-finetuned-libero-spatial": dict(
        revision="962318cec55ac10993ff0f5f43eda9a270b4c873", unnorm_key="libero_spatial",
        action_space="delta_eef_robosuite_osc"),
    "openvla/openvla-7b-finetuned-libero-goal": dict(
        revision="fa5ae1e7509348889295bba8e08621d8b55e9baf", unnorm_key="libero_goal",
        action_space="delta_eef_robosuite_osc"),
    "openvla/openvla-7b-finetuned-libero-10": dict(
        revision="80970322773f81baa2e22fe495d0487b93a05cfa", unnorm_key="libero_10",
        action_space="delta_eef_robosuite_osc"),
}

ACTION_DIM = 7
INPUT_SIZE = 224
CENTER_CROP_SCALE = 0.9  # area fraction, as in openvla_utils.get_vla_action
MAX_IMAGE_SIDE = 1024
IMAGE_KEYS = ("overhead_left", "overhead_right", "wrist")
PINNED_DEPENDENCIES = {"transformers": {"4.40.1"}, "tokenizers": {"0.19.1"},
                       "timm": {"0.9.10", "0.9.11", "0.9.12", "0.9.16"}}


def _download_pinned(checkpoint, revision, local_dir):
    """Materialise the exact pinned HF revision as a real directory (a repo-id
    load would resolve `main`)."""
    from huggingface_hub import snapshot_download

    target = Path(local_dir) / checkpoint.replace("/", "--") / revision
    return snapshot_download(repo_id=checkpoint, revision=revision, local_dir=str(target))


def _fetch_code_file(code_repo, filename, code_revision):
    from huggingface_hub import hf_hub_download

    return hf_hub_download(repo_id=code_repo, filename=filename, revision=code_revision)


def _localize_auto_map(auto_map, code_repo, path):
    localized = {}
    for name, ref in auto_map.items():
        if not isinstance(ref, str):
            raise RuntimeError(f"{path}: unexpected auto_map entry {name}={ref!r}")
        repo, sep, local_ref = ref.rpartition("--")
        if sep and repo != code_repo:
            raise RuntimeError(f"{path}: auto_map {name} points at {repo!r}, not the reviewed {code_repo!r}")
        if local_ref.split(".", 1)[0] not in _LOCAL_MODULES:
            raise RuntimeError(f"{path}: auto_map {name}={ref!r} is not one of {_LOCAL_MODULES}")
        localized[name] = local_ref
    return localized


def _localize_remote_code(model_dir, code_repo=CODE_REPO, code_revision=CODE_REVISION):
    """Make `model_dir` self-contained: write the reviewed OpenVLA remote code
    into it (overwriting any checkpoint-shipped copy) and point every `auto_map`
    at those local modules. Refuses an `auto_map` naming any other repo."""
    model_dir = Path(model_dir)
    pending = {}
    for name in AUTO_MAP_FILES:
        path = model_dir / name
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        if isinstance(data.get("auto_map"), dict):
            data["auto_map"] = _localize_auto_map(data["auto_map"], code_repo, path)
            pending[path] = data
    for filename in REMOTE_CODE_FILES:
        shutil.copyfile(_fetch_code_file(code_repo, filename, code_revision), model_dir / filename)
    for path, data in pending.items():
        path.write_text(json.dumps(data, indent=2) + "\n")


def _load_policy(model_dir, attn_implementation):
    """The official HF AutoClass load (README quick start / deploy.py)."""
    import torch
    from transformers import AutoModelForVision2Seq, AutoProcessor

    device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
    processor = AutoProcessor.from_pretrained(model_dir, trust_remote_code=True)
    vla = AutoModelForVision2Seq.from_pretrained(
        model_dir, attn_implementation=attn_implementation, torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True, trust_remote_code=True,
    ).to(device)
    vla.eval()
    return dict(vla=vla, processor=processor, device=device, dtype=torch.bfloat16)


def build_prompt(instruction):
    return f"In: What action should the robot take to {instruction.lower()}?\nOut:"


def preprocess_image(rgb, center_crop=True):
    """HxWx3 uint8 RGB -> 224x224 PIL RGB, optionally with OpenVLA's eval-time
    center crop (sqrt(CENTER_CROP_SCALE) of each side, resized back up)."""
    image = Image.fromarray(np.ascontiguousarray(rgb)).convert("RGB")
    image = image.resize((INPUT_SIZE, INPUT_SIZE), Image.LANCZOS)
    if center_crop:
        side = INPUT_SIZE * math.sqrt(CENTER_CROP_SCALE)
        offset = (INPUT_SIZE - side) / 2
        box = (offset, offset, offset + side, offset + side)
        image = image.resize((INPUT_SIZE, INPUT_SIZE), Image.BILINEAR, box=box)
    return image


class OpenVLAModel:
    def __init__(self, checkpoint=DEFAULT_CHECKPOINT, revision=None, unnorm_key=None,
                 local_dir=DEFAULT_LOCAL_DIR, center_crop=True, image_key=DEFAULT_IMAGE_KEY,
                 attn_implementation="sdpa"):
        if image_key not in IMAGE_KEYS:
            raise ValueError(f"image_key must be one of {IMAGE_KEYS}")
        profile = CHECKPOINT_PROFILES.get(checkpoint, {})
        if revision:
            model_dir = _download_pinned(checkpoint, revision, local_dir)
            _localize_remote_code(model_dir)
        else:
            model_dir = checkpoint  # explicit opt-out: remote code then resolves openvla/openvla-7b@main
        policy = _load_policy(model_dir, attn_implementation)
        self.vla = policy["vla"]
        self.processor = policy["processor"]
        self.device = policy["device"]
        self.dtype = policy["dtype"]

        keys = sorted(getattr(self.vla, "norm_stats", None) or {})
        if unnorm_key is None:
            unnorm_key = profile.get("unnorm_key")
        if unnorm_key is None:
            if len(keys) != 1:
                raise ValueError(f"{checkpoint} has unnorm keys {keys}; pass --unnorm-key to choose one")
            unnorm_key = keys[0]
        if unnorm_key not in keys:
            raise ValueError(f"unnorm key {unnorm_key!r} not in {checkpoint}'s norm_stats {keys}")
        action_stats = self.vla.norm_stats[unnorm_key]["action"]
        if len(action_stats.get("q01", [])) != ACTION_DIM:
            raise RuntimeError(f"{checkpoint}/{unnorm_key} action stats are not {ACTION_DIM}-D")
        self.unnorm_key = unnorm_key
        self.center_crop = center_crop
        self.image_key = image_key
        self.model_version = f"{checkpoint}@{revision[:12] if revision else 'unpinned'}"
        self.metadata = dict(
            checkpoint=checkpoint, revision=revision or "unpinned",
            code_repo=CODE_REPO, code_revision=CODE_REVISION if revision else "unpinned",
            unnorm_key=unnorm_key, action_space=profile.get("action_space", "unknown"),
            action_dim=ACTION_DIM, action_mask=list(action_stats.get("mask", [])),
            action_q01=list(action_stats["q01"]), action_q99=list(action_stats["q99"]),
            image_key=image_key, center_crop=center_crop, prompt=build_prompt("<instruction>"),
            returns="raw predict_action() output (one step, no chunk)",
        )

    def __call__(self, images, instruction):
        image = preprocess_image(images[self.image_key], self.center_crop)
        inputs = self.processor(build_prompt(instruction), image).to(self.device, dtype=self.dtype)
        action = self.vla.predict_action(**inputs, unnorm_key=self.unnorm_key, do_sample=False)
        values = np.asarray(action, dtype=np.float64).reshape(-1)
        if values.shape != (ACTION_DIM,):
            raise RuntimeError(f"model returned action of shape {values.shape}, expected ({ACTION_DIM},)")
        if not np.all(np.isfinite(values)):
            raise RuntimeError(f"model returned a non-finite action {values.tolist()}")
        return [float(v) for v in values]


def _decode_rgb(b64_png):
    """base64 PNG (OpenCV BGR-encoded by the client) -> HxWx3 uint8 RGB."""
    if not isinstance(b64_png, str):
        raise TypeError("image must be a base64 string")
    raw = base64.b64decode(b64_png, validate=True)
    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError("invalid PNG")
    if max(frame.shape[:2]) > MAX_IMAGE_SIDE:
        raise ValueError(f"image dimensions must be <= {MAX_IMAGE_SIDE}")
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def create_app(infer):
    from flask import Flask, jsonify, request
    app = Flask(__name__)
    app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024
    inference_lock = threading.Lock()
    image_key = getattr(infer, 'image_key', DEFAULT_IMAGE_KEY)

    @app.get('/health')
    def health():
        return jsonify(status='ready', protocol_version=1, model=infer.metadata)

    @app.post('/action')
    def action():
        if not inference_lock.acquire(blocking=False):
            return jsonify(error='openvla service busy'), 429
        try:
            body = request.get_json(silent=True)
            if not isinstance(body, dict):
                raise ValueError('expected a JSON object body')
            images_in = body['images']
            if not isinstance(images_in, dict):
                raise TypeError('images must be an object')
            images = {image_key: _decode_rgb(images_in[image_key])}
            instruction = body['instruction']
            if not isinstance(instruction, str) or not instruction.strip():
                raise ValueError('instruction must be a non-empty string')
            joint_state = body.get('joint_state')
            if joint_state is not None and not isinstance(joint_state, dict):
                raise TypeError('joint_state must be an object')
            started = time.perf_counter()
            result = infer(images, instruction)
            elapsed = (time.perf_counter() - started) * 1000
            return jsonify(action=[float(v) for v in result], model_version=infer.model_version,
                           inference_ms=elapsed)
        except (ValueError, KeyError, TypeError) as exc:
            return jsonify(error=str(exc)), 400
        except RuntimeError as exc:
            return jsonify(error=f'openvla inference failed: {exc}'), 500
        finally:
            inference_lock.release()
    return app


def _installed_versions():
    import importlib.metadata as metadata

    return {name: metadata.version(name) for name in PINNED_DEPENDENCIES}


def _check_dependency_versions():
    versions = _installed_versions()
    bad = {name: version for name, version in versions.items() if version not in PINNED_DEPENDENCIES[name]}
    if bad:
        raise SystemExit(
            f"OpenVLA needs transformers==4.40.1, tokenizers==0.19.1, timm 0.9.10-0.9.16 "
            f"(openvla README / modeling_prismatic.py); found {bad}. Use the isolated OpenVLA venv."
        )


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--checkpoint', default=DEFAULT_CHECKPOINT)
    parser.add_argument('--revision', default=None,
                        help="pinned HF commit sha (defaults to the reviewed profile's); "
                             "pass '' only to deliberately load the unpinned repo id")
    parser.add_argument('--unnorm-key', default=None)
    parser.add_argument('--local-dir', default=DEFAULT_LOCAL_DIR)
    parser.add_argument('--image-key', default=DEFAULT_IMAGE_KEY, choices=IMAGE_KEYS)
    parser.add_argument('--no-center-crop', action='store_true')
    parser.add_argument('--attn-implementation', default='sdpa', choices=('sdpa', 'eager', 'flash_attention_2'))
    parser.add_argument('--port', type=int, default=DEFAULT_PORT)
    args = parser.parse_args()
    revision = args.revision
    if revision is None:
        revision = CHECKPOINT_PROFILES.get(args.checkpoint, {}).get('revision')
        if revision is None:
            raise SystemExit(f"{args.checkpoint} has no reviewed profile; pass --revision <sha> "
                             "(or --revision '' to deliberately load it unpinned)")
    _check_dependency_versions()
    model = OpenVLAModel(args.checkpoint, revision=revision or None, unnorm_key=args.unnorm_key,
                         local_dir=args.local_dir, center_crop=not args.no_center_crop,
                         image_key=args.image_key, attn_implementation=args.attn_implementation)
    # CUDA kernels / lazy init happen before the socket opens and readiness is reported.
    model({model.image_key: np.zeros((INPUT_SIZE, INPUT_SIZE, 3), dtype=np.uint8)}, 'warm up')
    print(f'OpenVLA model loaded and warmed ({model.model_version}); ready', flush=True)
    create_app(model).run(host='0.0.0.0', port=args.port, threaded=True, use_reloader=False)


if __name__ == '__main__':
    main()


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = 'e5421b5fb3352e751db469fd3533340f1bc17ce12ea071cb6418caf1421c7bba'
actual = hashlib.sha256(Path('/content/openvla_server.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/openvla_server.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/openvla_server.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/openvla_server.py matches control/vla_bridge/openvla_server.py (sha256 {actual[:12]})')

## Launch

The launch cell first stops any earlier server: every process whose command line contains one of `openvla_server.py`, and whatever listens on port 3003.
It then starts this notebook's server as a separate OS process (unbuffered, log at `/content/openvla_server.log`).
It only succeeds once **its own** still-running process owns port 3003.
A stale server answering on the port is an error, never a successful launch (in the 2026-10-18 live session a stale stock server made a diffusion launch look successful).

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import start_server, stop_servers, wait_until_ready

PORT = 3003
OPENVLA_LOG = '/content/openvla_server.log'
STOP_PATTERNS = ['openvla_server.py']
OPENVLA_PYTHON = '/content/openvla_venv/bin/python'

stop_servers(PORT, STOP_PATTERNS)
openvla_server_process = start_server(
    [OPENVLA_PYTHON, '-u', '/content/openvla_server.py', '--port', '3003'],
    OPENVLA_LOG,
    env_overrides={'TOKENIZERS_PARALLELISM': 'false'},
)
health = wait_until_ready(
    openvla_server_process, PORT, OPENVLA_LOG,
    timeout_s=2400, health_url='http://127.0.0.1:3003/health',
)
print('OpenVLA adapter ready:', {k: health['model'][k] for k in ('checkpoint', 'revision', 'unnorm_key', 'action_space')})

## Open an HTTP tunnel to port 3003

In [ ]:
from pyngrok import ngrok

openvla_tunnel = ngrok.connect(3003, "http")  # plain JSON-over-HTTP
print(f"OpenVLA endpoint: {openvla_tunnel.public_url}")
SMOKE = ".venv/bin/python -m vla_bridge.backend_smoke --backend openvla --server-address https://<openvla_tunnel> --instruction 'Pick up the black cylinder and place it on the plate.'"
EPISODE = '.venv/bin/python run_vla_episode.py --backend openvla --server-address https://OPENVLA_TUNNEL.ngrok-free.app --instruction "Pick up the black cylinder and place it on the plate." --out outputs/openvla_UNIQUE_NAME --max-steps 20'
for placeholder in ('https://<openvla_tunnel>', 'https://OPENVLA_TUNNEL.ngrok-free.app'):
    SMOKE = SMOKE.replace(placeholder, openvla_tunnel.public_url)
    EPISODE = EPISODE.replace(placeholder, openvla_tunnel.public_url)
print('Local smoke (run in control/):')
print(SMOKE)
print('Local episode (run in control/):')
print(EPISODE)

## Run locally

Then, locally in `control/` (pass the base URL; the client appends `/action`):

```bash
.venv/bin/python run_vla_episode.py --backend openvla \
  --server-address https://OPENVLA_TUNNEL.ngrok-free.app \
  --instruction "Pick up the black cylinder and place it on the plate." \
  --out outputs/openvla_UNIQUE_NAME --max-steps 20
```

`--instruction` matches the default checkpoint's only training task. Each step's `model_version`
should read `toasta18/openvla-so101-cylinder@f00833ce5851`.

For a LIBERO alternate, add `--checkpoint openvla/openvla-7b-finetuned-libero-object`, the same
value the server was launched with. Its delta-EEF actions then go through
`eef_to_joint.solve_ik()`.

A step that logs `http-error-holding-position` means the adapter returned an error. Check
`/content/openvla_server.log`:
- 500 `openvla inference failed` means the model produced a non-finite or wrong-shaped action.
- 400 means a bad request.

Expect the default checkpoint's gripper to stay near ~53 %. That is a probable training-time
masking defect, explained in `openvla_server.py`.

Smoke run first (synthetic hardware, the arm cannot move; the tunnel cell prints it with the URL filled in):

```bash
.venv/bin/python -m vla_bridge.backend_smoke --backend openvla --server-address https://<openvla_tunnel> --instruction 'Pick up the black cylinder and place it on the plate.'
```

## Diagnostics

Run any of these after the launch cell, also from a restarted kernel: each cell re-declares
what it needs.
- Log: the tail of `/content/openvla_server.log`, then every line matching `ERROR`, `Error`, `Traceback`,
  `Exception`, `OutOfMemory`, `out of memory`, `Killed` or `beginAllocateToPool`.
- Processes: every known server process on this runtime (stale servers from other notebooks
  show up here); fails unless `openvla_server.py` owns port 3003.
- Package versions, GPU memory, and `/health`.

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import log_report

log_report('/content/openvla_server.log')

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import process_report

process_report(3003, 'openvla_server.py', globals().get('openvla_server_process'))

In [ ]:
import subprocess

OPENVLA_PYTHON = '/content/openvla_venv/bin/python'
probe = (
    "import importlib.metadata as m\n"
    "for p in ('torch', 'transformers', 'tokenizers', 'timm', 'accelerate', 'huggingface-hub'):\n"
    "    try:\n"
    "        print(p, m.version(p))\n"
    "    except m.PackageNotFoundError:\n"
    "        print(p, 'not installed')\n"
)
result = subprocess.run([OPENVLA_PYTHON, '-c', probe], capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-2000:])

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report

gpu_report()

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import health_report

health_report(3003)

## Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted server is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day: the cell below disconnects this notebook's tunnel, stops the
ngrok agent, stops the server and confirms port 3003 is free. Run it only
after the local episode or smoke run has finished.

In [ ]:
# OpenVLA teardown -- every session, after the local run has finished.
import sys

sys.path.insert(0, '/content')
from colab_server_utils import stop_server
from pyngrok import ngrok

if 'openvla_tunnel' in globals():
    try:
        ngrok.disconnect(openvla_tunnel.public_url)
        print(f'Disconnected {openvla_tunnel.public_url}')
    except Exception as exc:
        print(f'Tunnel already closed ({type(exc).__name__}: {exc})')
ngrok.kill()
print('ngrok agent stopped.')
stop_server(globals().get('openvla_server_process'), 3003, ['openvla_server.py'])